<a href="https://colab.research.google.com/github/juksentang/RVC-Retrieval-based-Voice-Conversion-Colab/blob/main/RVC_CLI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a> <a href="https://kaggle.com/kernels/welcome?src=https://github.com/juksentang/RVC-Retrieval-based-Voice-Conversion-Colab/blob/main/RVC_CLI.ipynb" target="_parent"><img src="https://kaggle.com/static/images/open-in-kaggle.svg" alt="Open In Kaggle"/></a>

# RVC for Colab / Kaggle（命令行版 / command-line）

在 Colab 或 Kaggle 上用命令行运行官方 [RVC-Project/Retrieval-based-Voice-Conversion-WebUI](https://github.com/RVC-Project/Retrieval-based-Voice-Conversion-WebUI)，训练模型并转换音频，不启动 WebUI。
Runs the official [RVC-Project/Retrieval-based-Voice-Conversion-WebUI](https://github.com/RVC-Project/Retrieval-based-Voice-Conversion-WebUI) from the command line on Colab or Kaggle: train a model and convert audio without starting the WebUI.

- **Colab**：选择 GPU 运行时（`代码执行程序 > 更改运行时类型 > T4 GPU`）。/ Select a GPU runtime (`Runtime > Change runtime type > T4 GPU`).
- **Kaggle**：在右侧设置中选择 `GPU T4 x2`，并打开 Internet（需要验证手机号）。数据集请先上传为 Kaggle Dataset 并添加到笔记本。/ In the side panel, select `GPU T4 x2` and turn Internet on (requires phone verification). Upload your dataset as a Kaggle Dataset and add it to the notebook.

需要 WebUI 的话请使用 [RVC_For_Colab.ipynb](https://github.com/juksentang/RVC-Retrieval-based-Voice-Conversion-Colab/blob/main/RVC_For_Colab.ipynb)（Colab 免费版可能不可用）。
For the WebUI, use [RVC_For_Colab.ipynb](https://github.com/juksentang/RVC-Retrieval-based-Voice-Conversion-Colab/blob/main/RVC_For_Colab.ipynb) instead (may not work on the free Colab tier).

项目主页 / Project page: https://github.com/juksentang/RVC-Retrieval-based-Voice-Conversion-Colab

# 准备环境 / Setup

In [ ]:
# @title 克隆 RVC 官方仓库 / Clone the official RVC repository
# @markdown 上游的 tag、分支或完整的提交 SHA，默认值为本笔记本适配的版本 / Upstream tag, branch or full commit SHA; the default is the version this notebook is written for
RVC_REF = "81eed5e8f68b6bed1789f682fe78cdd324495afc"  # @param {type:"string"}

import os
BASE = "/kaggle/temp" if os.path.isdir("/kaggle/input") else "/content"
ROOT = f"{BASE}/Retrieval-based-Voice-Conversion-WebUI"
PY = f"{BASE}/rvc-venv/bin/python"

import shutil, subprocess
shutil.rmtree(ROOT, ignore_errors=True)
os.makedirs(ROOT)
os.chdir(ROOT)
!git init -q
!git remote add origin https://github.com/RVC-Project/Retrieval-based-Voice-Conversion-WebUI.git
!git fetch -q --depth=1 origin "{RVC_REF}" && git checkout -q FETCH_HEAD
!git log -1 --format="%h %ad %s" --date=short


In [ ]:
# @title 安装依赖（独立的 Python 3.12 环境） / Install dependencies (separate Python 3.12 environment)

import os
BASE = "/kaggle/temp" if os.path.isdir("/kaggle/input") else "/content"
ROOT = f"{BASE}/Retrieval-based-Voice-Conversion-WebUI"
PY = f"{BASE}/rvc-venv/bin/python"

# RVC 2.3 需要 Python 3.12 与 numpy<2，而 Colab / Kaggle 的系统 Python 更新，所以单独建一个虚拟环境
# RVC 2.3 needs Python 3.12 and numpy<2, newer than the system Python on Colab / Kaggle, so it gets its own venv
import shutil
if shutil.which("uv") is None:
    !pip -q install uv
if not os.path.exists(PY):
    !uv venv -q --python 3.12 {BASE}/rvc-venv
!uv pip install -q --python {PY} torch==2.7.1 torchaudio==2.7.1 --index-url https://download.pytorch.org/whl/cu128
# 去掉 requirements 中的国内镜像，改用 PyPI / Drop the mirror pinned in the requirements file and use PyPI
!grep -v "^--index-url" {ROOT}/requirments_cu128_py312.txt > /tmp/rvc_requirements.txt
!uv pip install -q --python {PY} -r /tmp/rvc_requirements.txt
!{PY} -c "import sys, torch; print('Python', sys.version.split()[0], '| torch', torch.__version__, '| CUDA', torch.cuda.is_available())"


In [ ]:
# @title 下载模型 / Download models
# @markdown 下载 v1 底模（约 0.7 GB） / Also download v1 pretrained models (about 0.7 GB)
DOWNLOAD_V1 = False  # @param {type:"boolean"}
# @markdown 下载 PyMSS 人声分离模型（约 2 GB，WebUI 的人声分离页使用） / Download PyMSS vocal separation models (about 2 GB, used by the WebUI separation tab)
DOWNLOAD_PYMSS = False  # @param {type:"boolean"}

import os
BASE = "/kaggle/temp" if os.path.isdir("/kaggle/input") else "/content"
ROOT = f"{BASE}/Retrieval-based-Voice-Conversion-WebUI"
PY = f"{BASE}/rvc-venv/bin/python"

import zipfile
from huggingface_hub import hf_hub_download, snapshot_download

os.chdir(ROOT)
HF_REPO = "lj1995/VoiceConversionWebUI"
# WebUI 训练只提供 40k / 48k / The WebUI only trains at 40k and 48k
patterns = ["hubert_base/*", "pretrained_v2/*40k.pth", "pretrained_v2/*48k.pth"]
if DOWNLOAD_V1:
    patterns += ["pretrained/*40k.pth", "pretrained/*48k.pth"]
if DOWNLOAD_PYMSS:
    patterns.append("pymss_weights/*")
snapshot_download(HF_REPO, allow_patterns=patterns, local_dir="assets")
hf_hub_download(HF_REPO, "rmvpe.pt", local_dir="assets/rmvpe")
mute_zip = hf_hub_download(HF_REPO, "mute.zip", local_dir=f"{BASE}/.model-downloads")
with zipfile.ZipFile(mute_zip) as z:
    z.extractall("logs")
print("Done:", sorted(os.listdir("assets")), sorted(os.listdir("logs")))


In [ ]:
# @title 准备数据集 / Prepare the dataset
# @markdown zip 文件或文件夹，音频直接放在顶层。Colab 填谷歌云盘路径，Kaggle 填 /kaggle/input/... / A zip file or folder with the audio at the top level. Use a Google Drive path on Colab, or /kaggle/input/... on Kaggle
DATASET = "/content/drive/MyDrive/datasets/RVC/src.zip"  # @param {type:"string"}

import os
BASE = "/kaggle/temp" if os.path.isdir("/kaggle/input") else "/content"
ROOT = f"{BASE}/Retrieval-based-Voice-Conversion-WebUI"
PY = f"{BASE}/rvc-venv/bin/python"

import shutil, zipfile
if DATASET.startswith("/content/drive") and not os.path.isdir("/content/drive/MyDrive"):
    from google.colab import drive
    drive.mount("/content/drive")
DATA_DIR = f"{BASE}/dataset"
os.makedirs(DATA_DIR, exist_ok=True)
if DATASET.endswith(".zip"):
    with zipfile.ZipFile(DATASET) as z:
        z.extractall(DATA_DIR)
else:
    shutil.copytree(DATASET, DATA_DIR, dirs_exist_ok=True)
print(DATA_DIR, sorted(os.listdir(DATA_DIR))[:10])


In [ ]:
# @title 恢复备份的模型（首次使用无需执行） / Restore a backed-up model (skip on first use)
# @markdown 模型名 / Model name
MODELNAME = "lulu"  # @param {type:"string"}
# @markdown 要恢复的 epoch；“仅保存最新”时填 2333333 / Epoch to restore; use 2333333 if "only save latest" was on
MODELEPOCH = 2333333  # @param {type:"integer"}
# @markdown 备份目录，留空则 Colab 用谷歌云盘 MyDrive/RVC_backup，Kaggle 用 /kaggle/working/RVC_backup / Backup folder; leave empty for MyDrive/RVC_backup on Colab or /kaggle/working/RVC_backup on Kaggle
BACKUP_DIR = ""  # @param {type:"string"}
# @markdown 2025 年旧版笔记本的备份（文件在 MyDrive 根目录，且 G/D 文件名互换） / Backup made by the 2025 notebook (files in MyDrive root, G/D names swapped)
LEGACY_BACKUP = False  # @param {type:"boolean"}

import os
BASE = "/kaggle/temp" if os.path.isdir("/kaggle/input") else "/content"
ROOT = f"{BASE}/Retrieval-based-Voice-Conversion-WebUI"
PY = f"{BASE}/rvc-venv/bin/python"

if not BACKUP_DIR:
    BACKUP_DIR = "/kaggle/working/RVC_backup" if BASE.startswith("/kaggle") else "/content/drive/MyDrive/RVC_backup"
if BACKUP_DIR.startswith("/content/drive") and not os.path.isdir("/content/drive/MyDrive"):
    from google.colab import drive
    drive.mount("/content/drive")

import glob, shutil
os.chdir(ROOT)
exp_dir = f"logs/{MODELNAME}"
os.makedirs(exp_dir, exist_ok=True)
os.makedirs("assets/weights", exist_ok=True)

if LEGACY_BACKUP:
    src = "/content/drive/MyDrive"
    shutil.copy(f"{src}/{MODELNAME}_D_{MODELEPOCH}.pth", f"{exp_dir}/G_{MODELEPOCH}.pth")
    shutil.copy(f"{src}/{MODELNAME}_G_{MODELEPOCH}.pth", f"{exp_dir}/D_{MODELEPOCH}.pth")
    shutil.copy(f"{src}/{MODELNAME}{MODELEPOCH}.pth", f"assets/weights/{MODELNAME}.pth")
    extra = glob.glob(f"{src}/added_*{MODELNAME}*.index") + glob.glob(f"{src}/total_*.npy")
else:
    src = f"{BACKUP_DIR}/{MODELNAME}"
    for gd in ("G", "D"):
        shutil.copy(f"{src}/{gd}_{MODELEPOCH}.pth", exp_dir)
    for f in glob.glob(f"{src}/{MODELNAME}*.pth"):
        shutil.copy(f, "assets/weights")
    extra = glob.glob(f"{src}/*.index") + glob.glob(f"{src}/*.npy") + glob.glob(f"{src}/config.json")
for f in extra:
    shutil.copy(f, exp_dir)
print("Restored to", exp_dir, sorted(os.listdir(exp_dir)))


# 训练 / Training
所有单元格中的模型名、采样率和版本需保持一致。命令参考官方 [CLI 文档](https://github.com/RVC-Project/Retrieval-based-Voice-Conversion-WebUI/blob/main/docs/en/cli.md)。
Keep the model name, sample rate and version the same in every cell. The commands follow the official [CLI guide](https://github.com/RVC-Project/Retrieval-based-Voice-Conversion-WebUI/blob/main/docs/en/cli.md).

In [ ]:
# @title 音频预处理 / Preprocess audio
# @markdown 模型名 / Model name
MODELNAME = "lulu"  # @param {type:"string"}
# @markdown 采样率 / Sample rate
SAMPLERATE = "40k"  # @param ["40k", "48k"]
# @markdown 使用的进程数 / Number of processes
THREADCOUNT = 8  # @param {type:"integer"}

import os
BASE = "/kaggle/temp" if os.path.isdir("/kaggle/input") else "/content"
ROOT = f"{BASE}/Retrieval-based-Voice-Conversion-WebUI"
PY = f"{BASE}/rvc-venv/bin/python"

os.chdir(ROOT)
SR_HZ = int(SAMPLERATE[:-1]) * 1000
os.makedirs(f"logs/{MODELNAME}", exist_ok=True)
!{PY} train/preprocess.py {BASE}/dataset {SR_HZ} {THREADCOUNT} logs/{MODELNAME} False 3.7


In [ ]:
# @title 提取音高与特征 / Extract pitch and features
# @markdown 模型名 / Model name
MODELNAME = "lulu"  # @param {type:"string"}
# @markdown 版本 / Version
VERSION = "v2"  # @param ["v2", "v1"]
# @markdown 音高提取算法，2.3 训练只支持 rmvpe 和 pm，rmvpe_gpu 最快且效果最好 / Pitch extraction method; 2.3 only trains with rmvpe or pm, and rmvpe_gpu is the fastest and usually the best
ALGO = "rmvpe_gpu"  # @param ["rmvpe_gpu", "rmvpe", "pm"]
# @markdown 使用的进程数（rmvpe_gpu 不使用） / Number of processes (not used by rmvpe_gpu)
THREADCOUNT = 8  # @param {type:"integer"}

import os
BASE = "/kaggle/temp" if os.path.isdir("/kaggle/input") else "/content"
ROOT = f"{BASE}/Retrieval-based-Voice-Conversion-WebUI"
PY = f"{BASE}/rvc-venv/bin/python"

os.chdir(ROOT)
if ALGO == "rmvpe_gpu":
    !{PY} train/dataset/extract_f0.py cuda 1 0 0 logs/{MODELNAME} true
else:
    !{PY} train/dataset/extract_f0.py cpu logs/{MODELNAME} {THREADCOUNT} {ALGO}
!{PY} train/dataset/extract_hubert_feature.py cuda:0 1 0 0 logs/{MODELNAME} {VERSION} true


In [ ]:
# @title 训练模型 / Train the model
# @markdown 模型名 / Model name
MODELNAME = "lulu"  # @param {type:"string"}
# @markdown 采样率 / Sample rate
SAMPLERATE = "40k"  # @param ["40k", "48k"]
# @markdown 版本 / Version
VERSION = "v2"  # @param ["v2", "v1"]
# @markdown 批大小 / Batch size
BATCHSIZE = 8  # @param {type:"integer"}
# @markdown 总 epoch 数 / Total epochs
MODELEPOCH = 200  # @param {type:"integer"}
# @markdown 保存间隔（epoch） / Save every N epochs
EPOCHSAVE = 10  # @param {type:"integer"}
# @markdown 是否仅保存最新的 G/D 检查点（文件名为 G_2333333.pth） / Only keep the latest G/D checkpoint (saved as G_2333333.pth)
ONLYLATEST = True  # @param {type:"boolean"}
# @markdown 是否把训练集缓存到显存（数据集较小时可加速） / Cache the dataset in GPU memory (faster for small datasets)
CACHEDATA = False  # @param {type:"boolean"}
# @markdown 每次保存时都导出一个小模型到 assets/weights / Export a small model to assets/weights at every save
SAVEWEIGHTS = True  # @param {type:"boolean"}
# @markdown 在下方显示 TensorBoard / Show TensorBoard below
TENSORBOARD = True  # @param {type:"boolean"}

import os
BASE = "/kaggle/temp" if os.path.isdir("/kaggle/input") else "/content"
ROOT = f"{BASE}/Retrieval-based-Voice-Conversion-WebUI"
PY = f"{BASE}/rvc-venv/bin/python"

import json, random
os.chdir(ROOT)
# 与 webui.py 相同：生成 filelist.txt 和 config.json / Same as webui.py: write filelist.txt and config.json
exp_dir = f"{ROOT}/logs/{MODELNAME}"
fea_dim = 256 if VERSION == "v1" else 768
dirs = {"wav": f"{exp_dir}/0_gt_wavs", "fea": f"{exp_dir}/3_feature{fea_dim}",
        "f0": f"{exp_dir}/2a_f0", "f0nsf": f"{exp_dir}/2b-f0nsf"}
names = set.intersection(*(set(n.split(".")[0] for n in os.listdir(d)) for d in dirs.values()))
if not names:
    raise RuntimeError("没有可用于训练的音频，请先完成预处理和特征提取 / No usable samples, run preprocessing and feature extraction first")
lines = [f"{dirs['wav']}/{n}.wav|{dirs['fea']}/{n}.npy|{dirs['f0']}/{n}.wav.npy|{dirs['f0nsf']}/{n}.wav.npy|0" for n in names]
mute = f"{ROOT}/logs/mute"
lines += [f"{mute}/0_gt_wavs/mute{SAMPLERATE}.wav|{mute}/3_feature{fea_dim}/mute.npy|{mute}/2a_f0/mute.wav.npy|{mute}/2b-f0nsf/mute.wav.npy|0"] * 2
random.shuffle(lines)
with open(f"{exp_dir}/filelist.txt", "w", encoding="utf8") as f:
    f.write("\n".join(lines))
config_file = f"v1/{SAMPLERATE}.json" if VERSION == "v1" or SAMPLERATE == "40k" else f"v2/{SAMPLERATE}.json"
if not os.path.exists(f"{exp_dir}/config.json"):
    with open(f"configs/{config_file}", encoding="utf8") as f:
        cfg = json.load(f)
    with open(f"{exp_dir}/config.json", "w", encoding="utf8") as f:
        json.dump(cfg, f, ensure_ascii=False, indent=4, sort_keys=True)
        f.write("\n")
print(f"{len(names)} samples, config {config_file}")

if TENSORBOARD:
    try:
        get_ipython().run_line_magic("load_ext", "tensorboard")
        get_ipython().run_line_magic("tensorboard", f"--logdir {ROOT}/logs")
    except Exception as e:
        print("TensorBoard unavailable:", e)
PRETRAINED = "assets/pretrained_v2" if VERSION == "v2" else "assets/pretrained"
!{PY} train/train.py -e {MODELNAME} -sr {SAMPLERATE} -f0 1 -bs {BATCHSIZE} -g 0 -te {MODELEPOCH} -se {EPOCHSAVE} -pg {PRETRAINED}/f0G{SAMPLERATE}.pth -pd {PRETRAINED}/f0D{SAMPLERATE}.pth -l {int(ONLYLATEST)} -c {int(CACHEDATA)} -sw {int(SAVEWEIGHTS)} -v {VERSION}


In [ ]:
# @title 训练特征索引 / Train the feature index
# @markdown 模型名 / Model name
MODELNAME = "lulu"  # @param {type:"string"}
# @markdown 版本 / Version
VERSION = "v2"  # @param ["v2", "v1"]

import os
BASE = "/kaggle/temp" if os.path.isdir("/kaggle/input") else "/content"
ROOT = f"{BASE}/Retrieval-based-Voice-Conversion-WebUI"
PY = f"{BASE}/rvc-venv/bin/python"

os.chdir(ROOT)
!{PY} train/train_index.py {MODELNAME} {VERSION} assets/indices {os.cpu_count()} single


# 推理 / Inference

In [ ]:
# @title 转换音频 / Convert audio
# @markdown 模型名 / Model name
MODELNAME = "lulu"  # @param {type:"string"}
# @markdown 输入的音频文件或文件夹 / Input audio file or folder
INPUT = "/content/drive/MyDrive/datasets/RVC/input.wav"  # @param {type:"string"}
# @markdown 输出位置，留空则保存到 rvc_output 文件夹 / Output location; leave empty to save into the rvc_output folder
OUTPUT = ""  # @param {type:"string"}
# @markdown 变调（半音，男转女约 +12，女转男约 -12） / Pitch shift in semitones (about +12 for male to female, -12 for female to male)
PITCH = 0  # @param {type:"integer"}
# @markdown 音高提取算法 / Pitch extraction method
F0METHOD = "rmvpe"  # @param ["rmvpe", "pm"]
# @markdown 检索特征占比，0 表示不用索引 / Retrieval blend ratio; 0 disables the index
INDEXRATE = 0.75  # @param {type:"slider", min:0, max:1, step:0.05}
# @markdown 清辅音和呼吸声保护（0–0.5） / Consonant and breath protection (0–0.5)
PROTECT = 0.33  # @param {type:"slider", min:0, max:0.5, step:0.01}

import os
BASE = "/kaggle/temp" if os.path.isdir("/kaggle/input") else "/content"
ROOT = f"{BASE}/Retrieval-based-Voice-Conversion-WebUI"
PY = f"{BASE}/rvc-venv/bin/python"

if INPUT.startswith("/content/drive") and not os.path.isdir("/content/drive/MyDrive"):
    from google.colab import drive
    drive.mount("/content/drive")
os.chdir(ROOT)
if not OUTPUT:
    out_root = "/kaggle/working/rvc_output" if BASE.startswith("/kaggle") else f"{BASE}/rvc_output"
    os.makedirs(out_root, exist_ok=True)
    OUTPUT = out_root if os.path.isdir(INPUT) else f"{out_root}/{os.path.splitext(os.path.basename(INPUT))[0]}_{MODELNAME}.wav"
!{PY} infer/cli.py --model assets/weights/{MODELNAME}.pth --input "{INPUT}" --output "{OUTPUT}" --pitch {PITCH} --f0-method {F0METHOD} --index-rate {INDEXRATE} --protect {PROTECT} --overwrite
print("Output:", OUTPUT)


# 备份 / Backup

In [ ]:
# @title 备份模型 / Back up the model
# @markdown 模型名 / Model name
MODELNAME = "lulu"  # @param {type:"string"}
# @markdown 要备份的 epoch；“仅保存最新”时填 2333333 / Epoch to back up; use 2333333 if "only save latest" was on
MODELEPOCH = 2333333  # @param {type:"integer"}
# @markdown 备份目录，留空则 Colab 用谷歌云盘 MyDrive/RVC_backup，Kaggle 用 /kaggle/working/RVC_backup / Backup folder; leave empty for MyDrive/RVC_backup on Colab or /kaggle/working/RVC_backup on Kaggle
BACKUP_DIR = ""  # @param {type:"string"}

import os
BASE = "/kaggle/temp" if os.path.isdir("/kaggle/input") else "/content"
ROOT = f"{BASE}/Retrieval-based-Voice-Conversion-WebUI"
PY = f"{BASE}/rvc-venv/bin/python"

if not BACKUP_DIR:
    BACKUP_DIR = "/kaggle/working/RVC_backup" if BASE.startswith("/kaggle") else "/content/drive/MyDrive/RVC_backup"
if BACKUP_DIR.startswith("/content/drive") and not os.path.isdir("/content/drive/MyDrive"):
    from google.colab import drive
    drive.mount("/content/drive")

import glob, shutil
os.chdir(ROOT)
exp_dir = f"logs/{MODELNAME}"
dst = f"{BACKUP_DIR}/{MODELNAME}"
os.makedirs(dst, exist_ok=True)
files = [f"{exp_dir}/G_{MODELEPOCH}.pth", f"{exp_dir}/D_{MODELEPOCH}.pth", f"{exp_dir}/config.json"]
files += glob.glob(f"{exp_dir}/added_*.index") + glob.glob(f"{exp_dir}/total_*.npy")
files += glob.glob(f"assets/weights/{MODELNAME}*.pth")
for f in files:
    if os.path.exists(f):
        shutil.copy(f, dst)
        print("copied", f)
    else:
        print("missing", f)
print("Backup folder:", dst)
